# Q1 -- Which markets are the most efficient, and why?

One table, 33 rows, four measures. Self-contained: re-reads the parquet files, so
notebooks 1 and 2 do not have to be run first.

**Why these four measures**

| measure | why |
| --- | --- |
| time-weighted relative spread | raw cents cannot rank anything -- the tick is 1c and the book sits at 1 tick in 87% of rows, so median spread is 0.01 almost everywhere. Dividing by mid separates them. |
| time-weighted cost to cross | half-spread + the `p(1-p)*0.07` taker fee. The fee peaks at p=0.50 and vanishes at the extremes, so it reorders the ranking. |
| median top-5 depth (contracts) | every contract settles at $0 or $1, so face value is identical across all 33 markets and quantity is already the notional-equivalent. Unlike stocks, no need to multiply by price. |
| total volume traded (contracts) | quotes are cheap to post; volume is the evidence that anyone acted on them. |

**Time-weighted** means each quote is weighted by how long it stood before the next
message replaced it. The feed only sends a message when something changes, so rows are
events, not clock ticks, and message counts differ 12x across markets. An unweighted
average would partly be measuring how chatty each market's feed is.

In [1]:
import numpy as np
import pandas as pd

from utils import (time_weighted_relative_spread,
                   time_weighted_cost_to_cross,
                   median_total_quantity_top_5,
                   total_volume_traded)

df_trades = pd.read_parquet("data/trades_823753_pregame.parquet")
df_books  = pd.read_parquet("data/orderbook_data_823753_pregame.parquet")
for df in (df_trades, df_books):
    df["recv_ts_utc"] = pd.to_datetime(df["recv_ts_utc"], utc = True)

print("books ", df_books.shape)
print("trades", df_trades.shape)

books  (58275, 18)
trades (2232, 18)


In [2]:
# Apply the four metrics to each of the 33 markets in turn.
# mid and n_trades are carried along as context -- they are not efficiency measures,
# but the table cannot be read without knowing where each market sits on the ladder.
rows = []
for mkt in sorted(df_books["native_id"].unique()):
    b_m = df_books[df_books["native_id"] == mkt]
    t_m = df_trades[df_trades["native_id"] == mkt]
    rows.append({
        "market":       mkt.replace("-26AUG051940PITMIL", ""),   # strip the game stamp
        "mid":          ((b_m["best_bid"] + b_m["best_ask"]) / 2).median(),
        "rel_spread_%": 100 * time_weighted_relative_spread(b_m),
        "cost_cross_%": 100 * time_weighted_cost_to_cross(b_m),
        "depth_top5":   median_total_quantity_top_5(b_m),
        "volume":       total_volume_traded(t_m),
        "n_trades":     len(t_m),
    })

q1 = pd.DataFrame(rows).set_index("market")
q1.sort_values("volume", ascending = False).round(2)

,mid,rel_spread_%,cost_cross_%,depth_top5,volume,n_trades
market,,,,,,
KXMLBRFI,0.40,2.47,5.41,3232711.38,272584.21,1234
KXMLBTOTAL-8,0.44,2.27,5.06,444484.61,98745.90,385
KXMLBF5TOTAL-4,0.52,2.14,4.48,132730.82,35942.38,107
KXMLBTOTAL-7,0.57,1.75,3.86,340761.74,28881.61,71
KXMLBTEAMTOTAL-MIL4,0.50,3.01,5.06,28801.27,11829.93,70
KXMLBTOTAL-3,0.96,1.05,0.83,91512.52,7041.81,38
KXMLBF5TOTAL-5,0.36,2.70,5.78,44138.63,5928.34,24
KXMLBTOTAL-9,0.36,2.73,5.82,283777.11,4045.00,28
KXMLBTOTAL-6,0.66,1.53,3.14,190431.31,3289.39,71


### Ranked by each measure separately -- they disagree, which is the interesting part

In [3]:
for col, asc in [("rel_spread_%", True), ("cost_cross_%", True),
                 ("depth_top5", False), ("volume", False)]:
    top = q1.sort_values(col, ascending = asc).head(5)
    print(f"--- best 5 by {col} ---")
    print(top[[ "mid", col ]].round(2).to_string(), "\n")

--- best 5 by rel_spread_% ---
                 mid  rel_spread_%
market                            
KXMLBTOTAL-3    0.96          1.05
KXMLBTOTAL-4    0.86          1.17
KXMLBTOTAL-5    0.78          1.39
KXMLBF5TOTAL-3  0.66          1.50
KXMLBTOTAL-6    0.66          1.53 

--- best 5 by cost_cross_% ---
                      mid  cost_cross_%
market                                 
KXMLBTOTAL-3         0.96          0.83
KXMLBTOTAL-2         0.96          1.00
KXMLBTOTAL-4         0.86          1.59
KXMLBF5TOTAL-1       0.93          1.75
KXMLBTEAMTOTAL-MIL2  0.80          2.17 

--- best 5 by depth_top5 ---
               mid  depth_top5
market                        
KXMLBRFI      0.40  3232711.38
KXMLBTOTAL-8  0.44   444484.61
KXMLBTOTAL-7  0.57   340761.74
KXMLBTOTAL-9  0.36   283777.11
KXMLBTOTAL-6  0.66   190431.31 

--- best 5 by volume ---
                      mid     volume
market                              
KXMLBRFI             0.40  272584.21
KXMLBTOTAL-8         0.44

### All 33 markets on all 4 measures -- actual values

The top-5 lists above show who wins each measure but not where the other 28 markets
land. This is the full 33 x 4 table with the actual numbers, sorted by volume.

`mid` is carried along as orientation only -- it is not an efficiency measure, but both
cost columns depend on where a market sits on the price ladder, so the table cannot be
read without it.

Units: the two cost columns are percent of mid (lower is better). `depth_top5` and
`volume` are in the raw feed quantity unit (higher is better) -- see the To-do cell at
the bottom on why that unit is probably not "contracts".


In [4]:
pd.set_option("display.max_rows", 40)
pd.set_option("display.float_format", "{:,.2f}".format)

COLS = ["rel_spread_%", "cost_cross_%", "depth_top5", "volume"]

# Actual values, all 33 markets, no ranks. Sorted by volume so the busiest markets
# sit at the top and the long tail of near-untraded strikes is visible underneath.
full = q1[["mid"] + COLS].sort_values("volume", ascending = False)
full


,mid,rel_spread_%,cost_cross_%,depth_top5,volume
market,,,,,
KXMLBRFI,0.41,2.47,5.41,"3,232,711.38","272,584.21"
KXMLBTOTAL-8,0.45,2.27,5.06,"444,484.61","98,745.90"
KXMLBF5TOTAL-4,0.52,2.14,4.48,"132,730.82","35,942.38"
KXMLBTOTAL-7,0.57,1.75,3.86,"340,761.74","28,881.61"
KXMLBTEAMTOTAL-MIL4,0.49,3.01,5.06,"28,801.27","11,829.93"
KXMLBTOTAL-3,0.95,1.05,0.83,"91,512.52","7,041.81"
KXMLBF5TOTAL-5,0.36,2.70,5.78,"44,138.63","5,928.34"
KXMLBTOTAL-9,0.36,2.73,5.82,"283,777.11","4,045.00"
KXMLBTOTAL-6,0.66,1.53,3.14,"190,431.31","3,289.39"


### How far ahead is RFI, in magnitude?

Ranks say who is first. They do not say whether first is a nose ahead or a mile.
This compares RFI against the 33-market median and against the runner-up on each
measure.

In [5]:
# True = lower is better (the two cost measures), False = higher is better (liquidity)
LOWER_IS_BETTER = {"rel_spread_%": True, "cost_cross_%": True,
                   "depth_top5": False, "volume": False}

lead = pd.DataFrame({
    "RFI":            q1.loc["KXMLBRFI", COLS],
    "median of 33":   q1[COLS].median(),
    "2nd best":       [q1[c].nsmallest(2).iloc[-1] if LOWER_IS_BETTER[c]
                       else q1[c].nlargest(2).iloc[-1] for c in COLS],
})
lead["x median"]   = lead["RFI"] / lead["median of 33"]
lead["x 2nd best"] = lead["RFI"] / lead["2nd best"]

# For the cost measures, lower is better, so invert the ratio to read as "better by"
for c in ("rel_spread_%", "cost_cross_%"):
    lead.loc[c, ["x median", "x 2nd best"]] = 1 / lead.loc[c, ["x median", "x 2nd best"]]

print("liquidity measures: ratio > 1 means RFI is that many times better")
print("cost measures     : ratio inverted, so > 1 also means better\n")
lead.round(2)


liquidity measures: ratio > 1 means RFI is that many times better
cost measures     : ratio inverted, so > 1 also means better



,RFI,median of 33,2nd best,x median,x 2nd best
rel_spread_%,2.47,2.73,1.17,1.10,0.47
cost_cross_%,5.41,5.08,1.00,0.94,0.19
depth_top5,"3,232,711.38","30,449.63","444,484.61",106.17,7.27
volume,"272,584.21",524.87,"98,745.90",519.34,2.76


## Answer

**The most efficient market is `KXMLBRFI`** -- will there be at least 1 run in the
1st inning.

| | RFI | next best |
| --- | --- | --- |
| volume | 272,584 contracts | 98,746 (TOTAL-8) -- **2.8x** |
| trades | 1,234 | 385 (TOTAL-8) -- **3.2x** |
| top-5 depth | 3,232,711 contracts | 444,485 (TOTAL-8) -- **7.3x** |
| relative spread | 2.47% | mid-pack |

Depth and volume are not close. RFI is quoted seven times deeper than any other market
and trades nearly three times as much as the next one.

**Why it is the most efficient.** It is the only one of the 33 that is a standalone
binary rather than one strike on a ladder -- there is a single number to have an opinion
about, not a distribution to price consistently. It also resolves first, in the top of
the 1st inning, so capital is committed for the shortest time and the payoff is nearest.
Both effects concentrate attention and capital in one contract, and that is what
efficiency is: many participants pricing the same simple question.

**Its relative spread is only mid-pack, and that is the honest caveat.** At 2.47% it is
beaten by TEAMTOTAL-PIT4 (2.36%) -- a market that traded 525 contracts against RFI's
272,584. Tight quotes are cheap to post. Volume and depth are what show that the quotes
were real.

### The ranking inverts on cost to cross, and the reason is the fee

The cheapest market to trade is **TOTAL-3 at 0.83% of mid**, with TOTAL-2 at 1.00% --
both trading near 0.96. That is not good market-making, it is the fee formula: at
p = 0.96, `p(1-p)*0.07` is 0.27 cents, while at p = 0.50 it is 1.75 cents, or 3.5x the
half-spread. Contracts priced near certainty are cheap to cross because the exchange
barely charges for them.

Neither traded much -- 38 and 6 trades. So "cheapest to cross" and "most efficient" are
different questions, and a ranking built on spread alone would have put the wrong
markets on top.

### Least efficient

The extreme TEAMTOTAL strikes. `TEAMTOTAL-MIL8` (mid 0.10) has a **19.5%** relative
spread and one trade; `PIT8` (mid 0.08) is 12.8% with one trade; `PIT6` and `PIT7` are
quoted continuously and **never trade at all**. Low-probability strikes on a single
team's run total attract no attention, so the quotes stay wide and nothing crosses.

### To do

**Extensions, not yet done**

- Speed of information incorporation: when a large trade hits one strike, how many
  seconds until neighbouring strikes in the same chain reprice? Tests whether prices
  actually absorb information rather than merely looking tidy. Best remaining Q1 idea.
- No-arbitrage violations net of fees: within-chain monotonicity, non-negative
  butterflies, RFI <= F5 <= TOTAL nesting.

---

**Fix before submitting: `qty` is probably not "contracts"**

The write-up above calls `qty` a contract count. That is not established. Testing it:

| test | share passing |
| --- | --- |
| trade `qty` is an integer | 41.7% |
| `qty / price` is an integer | 9.8% |
| `qty / (1-price)` is an integer | 6.1% |
| **`qty * 100` is an integer** | **100.0%** |

Sample trades: `0.58 x 16.74`, `0.37 x 18.56`, `0.44 x 205.11`. Sample book levels:
`14419.24`, `29742.23`. Every quantity is exact to two decimal places and nothing else
fits. Prediction-market contracts trade in whole units, so these are not contract
counts. Given the `river_id` and `collector_run_id` columns this looks like a vendor
feed that rescales or aggregates, but the unit is unidentified.

What this does and does not break:

- **The ranking is unaffected.** Whatever the unit, it is consistent across all 33
  markets, and RFI leads depth by 5-7x. No rescaling flips that.
- **The justification for using raw quantity is weaker than written.** The argument
  above -- "every contract settles at $0 or $1, so face value is identical across
  markets and quantity is already the notional-equivalent" -- depends on these being
  contracts. If `qty` is dollar-denominated, that reasoning inverts.

Fix: state the units conditionally rather than asserting "contracts", and keep the
contracts-vs-notional argument as the reasoning that *would* apply if confirmed.

---

**Add as a cell: the chain-aggregation check (already run, result below)**

The obvious objection to "RFI is the most efficient" is that RFI is a single market
competing against chains whose attention is split across many strikes. Aggregating by
chain answers it:

| chain | volume | trades | strikes |
| --- | --- | --- | --- |
| KXMLBRFI | 272,584 | 1,234 | 1 |
| KXMLBTOTAL | 150,770 | 690 | 11 |
| KXMLBF5TOTAL | 42,800 | 160 | 7 |
| KXMLBTEAMTOTAL | 15,951 | 148 | 14 |

RFI alone beats all eleven TOTAL strikes combined by 1.8x. Worth showing, since it
pre-empts the main line of attack.

**Two other checks already run, both passed:**

- Cost to cross priced from the sell side instead of the buy side gives rank
  correlation 0.9993 and an identical top 5. The docstring claim that the sides mirror
  each other is verified, not assumed.
- Depth measured at top-of-book only (rather than summed across 5 levels) keeps the
  same winner: RFI 353,691 vs TOTAL-8 at 69,006, a 5.1x lead instead of 7.3x. The
  depth result is not an artifact of large size resting far from the touch.

---

**Minor inconsistencies, cosmetic**

- The `mid` column is a plain median while the metrics beside it are time-weighted.
  It is only there for orientation, but it is inconsistent.
- `median_total_quantity_top_5` is event-weighted (a median over rows) while the
  spread metrics are time-weighted. Documented in the docstring; defensible because a
  median depends only on ordering, but be ready to defend it.